# 16 — v16: CatBoost as 4th Base Model on v15 Feature Set

**Current best:** v15 → OOF RMSE **$21,593** | Goal: combine v15 features + CatBoost diversity

v12 showed CatBoost added **$49** over v9. v15 added **$34** over v9.  
This notebook combines both on the same feature set for the first time.

---
## 1. Imports & Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from math import radians, cos, sin, asin, sqrt

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, make_scorer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

train = pd.read_csv('../../data/raw/train.csv', low_memory=False)
test  = pd.read_csv('../../data/raw/test.csv',  low_memory=False)
print(f'Train: {train.shape}  |  Test: {test.shape}')

Train: (150634, 77)  |  Test: (16737, 76)


---
## 2. Feature Engineering

Same as v8, plus three new features:
- **`postal_sector`** — first 4 chars of `postal` (598 unique sectors, 100% test coverage)
- **`block_num`** — numeric part of `block` string (e.g. "123A" → 123)
- `postal` and `block` (raw) will be dropped in Section 4

In [2]:
DROP_COLS   = ['floor_area_sqft','lower','upper','mid','full_flat_type',
               'address','Tranc_YearMonth','residential','year_completed']
SOLD_COLS   = ['1room_sold','2room_sold','3room_sold','4room_sold',
               '5room_sold','exec_sold','multigen_sold','studio_apartment_sold']
RENTAL_COLS = ['1room_rental','2room_rental','3room_rental','other_room_rental']
FILL_ZERO   = ['Hawker_Within_500m','Mall_Within_500m','Hawker_Within_1km',
               'Mall_Within_1km','Hawker_Within_2km','Mall_Within_2km']
MATURE_ESTATES = {
    'ANG MO KIO','BEDOK','BISHAN','BUKIT MERAH','BUKIT TIMAH',
    'CENTRAL AREA','CLEMENTI','GEYLANG','KALLANG/WHAMPOA',
    'MARINE PARADE','PASIR RIS','QUEENSTOWN','SERANGOON','TAMPINES','TOA PAYOH'
}
ROOM_COUNT = {'1 ROOM':1,'2 ROOM':2,'3 ROOM':3,'4 ROOM':4,
              '5 ROOM':5,'EXECUTIVE':5,'MULTI-GENERATION':6}
FLAT_TYPE_RANK = {'1 ROOM':1,'2 ROOM':2,'3 ROOM':3,'4 ROOM':4,
                  '5 ROOM':5,'EXECUTIVE':6,'MULTI-GENERATION':7}
CBD_LAT, CBD_LON = 1.2847, 103.8510

STREET_FREQ = train['street_name'].value_counts().to_dict()

def haversine_km(lat, lon, lat2=CBD_LAT, lon2=CBD_LON):
    R = 6371
    lat, lon, lat2, lon2 = map(radians, [lat, lon, lat2, lon2])
    a = sin((lat2-lat)/2)**2 + cos(lat)*cos(lat2)*sin((lon2-lon)/2)**2
    return 2*R*asin(sqrt(a))

def engineer_features(df, amenity_caps=None):
    df = df.copy()
    for c in FILL_ZERO:
        df[c] = df[c].fillna(0)

    df['remaining_lease']  = 99 - (df['Tranc_Year'] - df['lease_commence_date'])
    df['dist_to_cbd']      = df.apply(lambda r: haversine_km(r['Latitude'], r['Longitude']), axis=1)
    df['is_mature_estate'] = df['town'].str.upper().isin(MATURE_ESTATES).astype(int)
    df['tranc_month_sin']     = np.sin(2*np.pi*df['Tranc_Month']/12)
    df['tranc_month_cos']     = np.cos(2*np.pi*df['Tranc_Month']/12)
    df['total_sold']          = df[SOLD_COLS].sum(axis=1)
    df['total_rental']        = df[RENTAL_COLS].sum(axis=1)
    df['rental_ratio']        = (df['total_rental'] / df['total_dwelling_units'].replace(0, np.nan)).fillna(0)
    df['num_rooms']           = df['flat_type'].str.upper().map(ROOM_COUNT).fillna(4)
    df['floor_area_per_room'] = df['floor_area_sqm'] / df['num_rooms']
    caps = amenity_caps or {}
    new_caps = {}
    for col in ['mrt_nearest_distance','Mall_Nearest_Distance','Hawker_Nearest_Distance']:
        cap = caps.get(col, df[col].quantile(0.99))
        new_caps[col] = cap
        inv = 1 / df[col].clip(1, cap)
        inv_min, inv_max = inv.min(), inv.max()
        df[f'_inv_{col}'] = (inv - inv_min) / (inv_max - inv_min + 1e-9)
    df['amenity_score'] = (df['_inv_mrt_nearest_distance'] +
                           df['_inv_Mall_Nearest_Distance'] +
                           df['_inv_Hawker_Nearest_Distance']) / 3
    df.drop(columns=[c for c in df.columns if c.startswith('_inv_')], inplace=True)
    df['dist_x_storey']   = df['dist_to_cbd'] * df['mid_storey']
    df['lease_x_area']    = df['remaining_lease'] * df['floor_area_sqm']
    df['log_dist_to_cbd'] = np.log1p(df['dist_to_cbd'])
    df['street_freq']     = df['street_name'].map(STREET_FREQ).fillna(0)
    df['postal_sector']   = df['postal'].astype(str).str[:4]
    df['block_num']       = pd.to_numeric(
        df['block'].astype(str).str.extract(r'(\d+)')[0], errors='coerce'
    ).fillna(0)

    # From v14b — Remaining lease non-linearity
    df['log_remaining_lease'] = np.log1p(df['remaining_lease'])
    df['lease_below_60']      = (df['remaining_lease'] < 60).astype(int)
    df['lease_x_below60']     = df['remaining_lease'] * df['lease_below_60']

    # From v14b — Storey x flat_type interaction
    df['flat_type_rank']    = df['flat_type'].str.upper().map(FLAT_TYPE_RANK).fillna(3)
    df['storey_x_flattype'] = df['mid_storey'] * df['flat_type_rank']

    # NEW v15 — Group key for (town, flat_type) OOF encoding (dropped from X)
    df['town_flat_type'] = df['town'].astype(str) + '_' + df['flat_type'].astype(str)

    return df, new_caps

train_fe, train_caps = engineer_features(train)
test_fe,  _          = engineer_features(test, amenity_caps=train_caps)
print(f'Train: {train_fe.shape}  |  Test: {test_fe.shape}')
print(f'town_flat_type groups: {train_fe["town_flat_type"].nunique()} (target ~182)')

Train: (150634, 100)  |  Test: (16737, 99)
town_flat_type groups: 128 (target ~182)


---
## 3. Per-Fold OOF Target Encoding

Generalised helper replaces `oof_town_median()` from v8. Apply to:
- `town` → `town_median_price` (as before)
- `postal_sector` → `postal_sector_median_price` (**new**, biggest expected gain)
- `flat_model` → `flat_model_median_price` (**new**)

> **No leakage**: each val row's encoded value is computed from the other 4 folds only.

In [3]:
def oof_group_median(group_series, y_series, n_splits=5, random_state=42):
    """OOF median target encoding for any group column."""
    groups = group_series.values
    y      = y_series.values
    encoded   = np.zeros(len(groups))
    global_med = np.median(y)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for tr_idx, va_idx in kf.split(groups):
        fold_map = {}
        for g, p in zip(groups[tr_idx], y[tr_idx]):
            fold_map.setdefault(g, []).append(p)
        fold_map = {g: np.median(ps) for g, ps in fold_map.items()}
        for i in va_idx:
            encoded[i] = fold_map.get(groups[i], global_med)
    return encoded

global_price_med = train['resale_price'].median()

# --- Training set: OOF encoding (no leakage) ---
train_fe['town_median_price']           = oof_group_median(train_fe['town'],           train['resale_price'])
train_fe['postal_sector_median_price']  = oof_group_median(train_fe['postal_sector'],  train['resale_price'])
train_fe['flat_model_median_price']     = oof_group_median(train_fe['flat_model'],     train['resale_price'])
train_fe['town_flat_type_median_price'] = oof_group_median(train_fe['town_flat_type'], train['resale_price'])  # NEW v15
train_fe['year_median_price']           = oof_group_median(train_fe['Tranc_Year'],     train['resale_price'])  # NEW v15

# --- Test set: full-training-set maps (no leakage; test labels unknown) ---
_tmp = pd.DataFrame({
    'town':           train_fe['town'].values,
    'postal_sector':  train_fe['postal_sector'].values,
    'flat_model':     train_fe['flat_model'].values,
    'town_flat_type': train_fe['town_flat_type'].values,
    'Tranc_Year':     train_fe['Tranc_Year'].values,
    'resale_price':   train['resale_price'].values,
})
full_town_map   = _tmp.groupby('town')['resale_price'].median()
full_sector_map = _tmp.groupby('postal_sector')['resale_price'].median()
full_model_map  = _tmp.groupby('flat_model')['resale_price'].median()
full_tf_map     = _tmp.groupby('town_flat_type')['resale_price'].median()  # NEW v15
full_year_map   = _tmp.groupby('Tranc_Year')['resale_price'].median()      # NEW v15

test_fe['town_median_price']           = test_fe['town'].map(full_town_map).fillna(global_price_med)
test_fe['postal_sector_median_price']  = test_fe['postal_sector'].map(full_sector_map).fillna(global_price_med)
test_fe['flat_model_median_price']     = test_fe['flat_model'].map(full_model_map).fillna(global_price_med)
test_fe['town_flat_type_median_price'] = test_fe['town_flat_type'].map(full_tf_map).fillna(global_price_med)    # NEW v15
test_fe['year_median_price']           = test_fe['Tranc_Year'].map(full_year_map).fillna(global_price_med)      # NEW v15

print('OOF encoding done: 5 encoded columns')
print(f'town_flat_type_median_price nulls — train: {train_fe["town_flat_type_median_price"].isna().sum()}, test: {test_fe["town_flat_type_median_price"].isna().sum()}')
print(f'year_median_price           nulls — train: {train_fe["year_median_price"].isna().sum()}, test: {test_fe["year_median_price"].isna().sum()}')

OOF encoding done: 5 encoded columns
town_flat_type_median_price nulls — train: 0, test: 0
year_median_price           nulls — train: 0, test: 0


---
## 4. Prepare X and y

`postal` and `block` (raw, high-cardinality) are dropped — replaced by `postal_sector_median_price` and `block_num`.

In [4]:
DROP_ALL = (['id', 'resale_price', 'postal', 'block', 'town_flat_type']
            + DROP_COLS + SOLD_COLS + RENTAL_COLS + ['num_rooms'])

X      = train_fe.drop(columns=DROP_ALL, errors='ignore')
y      = train['resale_price'].values
y_log  = np.log1p(y)

X_test = test_fe.drop(columns=[c for c in DROP_ALL if c != 'resale_price'], errors='ignore')
X_test = X_test.reindex(columns=X.columns, fill_value=0)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    X[col]      = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print(f'Features: {X.shape[1]}  (num={len(num_cols)}, cat={len(cat_cols)})')
new_v15 = [c for c in ['town_flat_type_median_price','year_median_price'] if c in num_cols]
print(f'New v15 features in X: {new_v15}')
print(f'town_flat_type correctly dropped: {"town_flat_type" not in X.columns}')
assert X_test.shape[1] == X.shape[1], 'X/X_test column mismatch'

Features: 78  (num=63, cat=15)
New v15 features in X: ['town_flat_type_median_price', 'year_median_price']
town_flat_type correctly dropped: True


---
## 5. Preprocessing Pipeline

In [5]:
num_pipe = Pipeline([('imp', SimpleImputer(strategy='median'))])
cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def neg_dollar_rmse(y_log_true, y_log_pred):
    return -np.sqrt(mean_squared_error(np.expm1(y_log_true), np.expm1(y_log_pred)))

dollar_rmse_scorer = make_scorer(neg_dollar_rmse)
print('Preprocessor ready.')

Preprocessor ready.


---
## 6. RandomizedSearchCV — XGBoost

Wider search space than v8 (40 trials vs 30, more parameter options) using the same `dollar_rmse_scorer` (RMSE in SGD, not log-space).

**Key additions vs v8 grid:**
- `n_estimators` extended to 2000
- `reg_alpha` and `reg_lambda` now explicitly searched (v8 had fixed defaults)
- `colsample_bytree` includes 0.4–0.5 range (sometimes helps with many features)

In [6]:
# Reuse v9's best params — 5 new scalar features won't shift the optimal region
XGB_PARAMS = {
    'n_estimators': 2000, 'max_depth': 7, 'learning_rate': 0.05,
    'subsample': 0.9, 'colsample_bytree': 0.4, 'min_child_weight': 7,
    'reg_alpha': 0.01, 'reg_lambda': 1.5,
    'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist'
}
print('XGB_PARAMS set (v9 best):', XGB_PARAMS)

XGB_PARAMS set (v9 best): {'n_estimators': 2000, 'max_depth': 7, 'learning_rate': 0.05, 'subsample': 0.9, 'colsample_bytree': 0.4, 'min_child_weight': 7, 'reg_alpha': 0.01, 'reg_lambda': 1.5, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist'}


---
## 7. RandomizedSearchCV — LightGBM

Extended grid with `num_leaves` up to 300 and explicit regularisation search.

In [7]:
LGBM_PARAMS = {
    'n_estimators': 1000, 'max_depth': 12, 'num_leaves': 300, 'learning_rate': 0.03,
    'subsample': 0.8, 'colsample_bytree': 0.5, 'min_child_samples': 20,
    'reg_alpha': 0, 'reg_lambda': 0.5,
    'random_state': 42, 'n_jobs': -1, 'verbosity': -1
}
print('LGBM_PARAMS set (v9 best):', LGBM_PARAMS)

LGBM_PARAMS set (v9 best): {'n_estimators': 1000, 'max_depth': 12, 'num_leaves': 300, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.5, 'min_child_samples': 20, 'reg_alpha': 0, 'reg_lambda': 0.5, 'random_state': 42, 'n_jobs': -1, 'verbosity': -1}


---
## 8. RandomizedSearchCV — Extra Trees

ET has no learning rate interaction so Bayesian search adds little. Keep RandomizedSearchCV.

In [8]:
ET_PARAMS = {
    'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 1,
    'max_features': 0.8, 'max_depth': 20,
    'random_state': 42, 'n_jobs': -1
}
print('ET_PARAMS set (v9 best):', ET_PARAMS)

# CatBoost — v12 best params (symmetric trees, different regularisation from XGB/LGBM)
CAT_PARAMS = {
    'iterations': 1000,
    'depth': 7,
    'learning_rate': 0.08,
    'l2_leaf_reg': 1,
    'colsample_bylevel': 0.7,
    'min_child_samples': 5,
    'loss_function': 'RMSE',
    'random_seed': 42,
    'verbose': 0
}
print('CAT_PARAMS set (v12 best):', CAT_PARAMS)

ET_PARAMS set (v9 best): {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.8, 'max_depth': 20, 'random_state': 42, 'n_jobs': -1}
CAT_PARAMS set (v12 best): {'iterations': 1000, 'depth': 7, 'learning_rate': 0.08, 'l2_leaf_reg': 1, 'colsample_bylevel': 0.7, 'min_child_samples': 5, 'loss_function': 'RMSE', 'random_seed': 42, 'verbose': 0}


---
## 9. Generate OOF Predictions (5-Fold) — 4 Models

Per-fold target encoding is **recomputed inside each fold** from training rows only.  
CatBoost added as 4th base alongside XGB, LGBM, and ExtraTrees.

In [9]:
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)

xgb_oof  = np.zeros(len(X))
lgbm_oof = np.zeros(len(X))
et_oof   = np.zeros(len(X))
cat_oof  = np.zeros(len(X))

xgb_test_folds  = np.zeros((5, len(X_test)))
lgbm_test_folds = np.zeros((5, len(X_test)))
et_test_folds   = np.zeros((5, len(X_test)))
cat_test_folds  = np.zeros((5, len(X_test)))

fold_rmses_xgb  = []
fold_rmses_lgbm = []
fold_rmses_et   = []
fold_rmses_cat  = []

# (group_col, price_col, min_count) — all accessed via train_fe (retains all cols)
ENCODE_PAIRS = [
    ('town',           'town_median_price',           1),
    ('postal_sector',  'postal_sector_median_price',  1),
    ('flat_model',     'flat_model_median_price',     1),
    ('town_flat_type', 'town_flat_type_median_price', 3),  # v15
    ('Tranc_Year',     'year_median_price',           1),  # v15
]

print('Generating OOF predictions (5-fold, 4 models)...')
print(f'{"Fold":>5}  {"XGB RMSE":>12}  {"LGBM RMSE":>12}  {"ET RMSE":>12}  {"CAT RMSE":>12}')
print('-' * 69)

for fold, (tr_idx, va_idx) in enumerate(kf5.split(X)):
    X_tr = X.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()
    y_tr, y_va = y[tr_idx], y[va_idx]
    y_tr_log   = np.log1p(y_tr)
    global_med_tr = np.median(y_tr)

    fe_tr = train_fe.iloc[tr_idx]  # always use train_fe for group col lookups
    fe_va = train_fe.iloc[va_idx]

    for group_col, price_col, min_ct in ENCODE_PAIRS:
        fold_map = {}
        for g, p in zip(fe_tr[group_col].values, y_tr):
            fold_map.setdefault(g, []).append(p)
        fold_map = {g: np.median(ps) for g, ps in fold_map.items() if len(ps) >= min_ct}
        X_tr[price_col] = fe_tr[group_col].map(fold_map).fillna(global_med_tr).values
        X_va[price_col] = fe_va[group_col].map(fold_map).fillna(global_med_tr).values

    xgb_pipe = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**XGB_PARAMS))])
    xgb_pipe.fit(X_tr, y_tr_log)
    xgb_oof[va_idx]      = xgb_pipe.predict(X_va)
    xgb_test_folds[fold] = xgb_pipe.predict(X_test)
    fold_rmses_xgb.append(rmse(y_va, np.expm1(xgb_oof[va_idx])))

    lgbm_pipe = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])
    lgbm_pipe.fit(X_tr, y_tr_log)
    lgbm_oof[va_idx]      = lgbm_pipe.predict(X_va)
    lgbm_test_folds[fold] = lgbm_pipe.predict(X_test)
    fold_rmses_lgbm.append(rmse(y_va, np.expm1(lgbm_oof[va_idx])))

    et_pipe = Pipeline([('pre', preprocessor), ('model', ExtraTreesRegressor(**ET_PARAMS))])
    et_pipe.fit(X_tr, y_tr_log)
    et_oof[va_idx]      = et_pipe.predict(X_va)
    et_test_folds[fold] = et_pipe.predict(X_test)
    fold_rmses_et.append(rmse(y_va, np.expm1(et_oof[va_idx])))

    cat_pipe = Pipeline([('pre', preprocessor), ('model', CatBoostRegressor(**CAT_PARAMS))])
    cat_pipe.fit(X_tr, y_tr_log)
    cat_oof[va_idx]      = cat_pipe.predict(X_va)
    cat_test_folds[fold] = cat_pipe.predict(X_test)
    fold_rmses_cat.append(rmse(y_va, np.expm1(cat_oof[va_idx])))

    print(f'{fold+1:>5}  ${fold_rmses_xgb[-1]:>10,.0f}  ${fold_rmses_lgbm[-1]:>10,.0f}  ${fold_rmses_et[-1]:>10,.0f}  ${fold_rmses_cat[-1]:>10,.0f}')

print('-' * 69)
print(f'{"Mean":>5}  ${np.mean(fold_rmses_xgb):>10,.0f}  ${np.mean(fold_rmses_lgbm):>10,.0f}  ${np.mean(fold_rmses_et):>10,.0f}  ${np.mean(fold_rmses_cat):>10,.0f}')

xgb_test_avg  = xgb_test_folds.mean(axis=0)
lgbm_test_avg = lgbm_test_folds.mean(axis=0)
et_test_avg   = et_test_folds.mean(axis=0)
cat_test_avg  = cat_test_folds.mean(axis=0)
print('\nOOF generation complete.')

Generating OOF predictions (5-fold, 4 models)...
 Fold      XGB RMSE     LGBM RMSE       ET RMSE      CAT RMSE
---------------------------------------------------------------------
    1  $    21,510  $    21,480  $    23,040  $    22,255
    2  $    22,278  $    22,248  $    23,697  $    23,062
    3  $    21,643  $    21,622  $    23,268  $    22,600
    4  $    21,899  $    21,853  $    23,499  $    22,565
    5  $    21,777  $    21,679  $    23,354  $    22,556
---------------------------------------------------------------------
 Mean  $    21,821  $    21,776  $    23,372  $    22,607

OOF generation complete.


---
## 10. Sanity Check — Individual Models & Blends

In [10]:
print('Individual OOF RMSE:')
print(f'  XGB:      ${rmse(y, np.expm1(xgb_oof)):>8,.0f}')
print(f'  LGBM:     ${rmse(y, np.expm1(lgbm_oof)):>8,.0f}')
print(f'  ET:       ${rmse(y, np.expm1(et_oof)):>8,.0f}')
print(f'  CatBoost: ${rmse(y, np.expm1(cat_oof)):>8,.0f}')

blend_3 = np.expm1((xgb_oof + lgbm_oof + et_oof) / 3)
blend_4 = np.expm1((xgb_oof + lgbm_oof + et_oof + cat_oof) / 4)
print(f'\n3-model equal-weight blend OOF RMSE: ${rmse(y, blend_3):>8,.0f}')
print(f'4-model equal-weight blend OOF RMSE: ${rmse(y, blend_4):>8,.0f}')
print(f'v15 OOF RMSE (3-model baseline):      ~$21,593')

Individual OOF RMSE:
  XGB:      $  21,823
  LGBM:     $  21,778
  ET:       $  23,373
  CatBoost: $  22,609

3-model equal-weight blend OOF RMSE: $  21,722
4-model equal-weight blend OOF RMSE: $  21,633
v15 OOF RMSE (3-model baseline):      ~$21,593


---
## 11. Ridge Meta-Model on 3 OOF Features

Ridge learns optimal weights for the 3 base model predictions.
Check that all 3 coefficients are positive — confirms each model contributes genuinely.

In [11]:
meta_X_train = np.column_stack([xgb_oof, lgbm_oof, et_oof, cat_oof])
meta_X_test  = np.column_stack([xgb_test_avg, lgbm_test_avg, et_test_avg, cat_test_avg])

print('Ridge alpha sweep (4 meta-features):')
print(f'{"alpha":>10}  {"RMSE":>12}  {"XGB coef":>10}  {"LGBM coef":>10}  {"ET coef":>10}  {"CAT coef":>10}')
print('-' * 74)

best_meta_rmse   = float('inf')
best_alpha_ridge = 1.0

for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(meta_X_train, y_log)
    meta_pred_log = ridge.predict(meta_X_train)
    meta_rmse_val = rmse(y, np.expm1(meta_pred_log))
    print(f'{alpha:>10.3f}  ${meta_rmse_val:>10,.0f}  {ridge.coef_[0]:>10.4f}  {ridge.coef_[1]:>10.4f}  {ridge.coef_[2]:>10.4f}  {ridge.coef_[3]:>10.4f}')
    if meta_rmse_val < best_meta_rmse:
        best_meta_rmse   = meta_rmse_val
        best_alpha_ridge = alpha

print(f'\nBest Ridge alpha: {best_alpha_ridge}')
meta_model = Ridge(alpha=best_alpha_ridge)
meta_model.fit(meta_X_train, y_log)
print(f'Learned coefficients:')
print(f'  XGB:      {meta_model.coef_[0]:.4f}')
print(f'  LGBM:     {meta_model.coef_[1]:.4f}')
print(f'  ET:       {meta_model.coef_[2]:.4f}')
print(f'  CatBoost: {meta_model.coef_[3]:.4f}')
print(f'  Intercept: {meta_model.intercept_:.4f}')
print(f'\nOOF meta-train RMSE (optimistic): ${best_meta_rmse:,.0f}')
print(f'v15 OOF RMSE for comparison:        $21,593')

Ridge alpha sweep (4 meta-features):
     alpha          RMSE    XGB coef   LGBM coef     ET coef    CAT coef
--------------------------------------------------------------------------
     0.001  $    21,552      0.3431      0.3673      0.1198      0.1725
     0.010  $    21,552      0.3431      0.3672      0.1198      0.1726
     0.100  $    21,552      0.3430      0.3666      0.1201      0.1729
     1.000  $    21,553      0.3418      0.3612      0.1233      0.1765
    10.000  $    21,558      0.3268      0.3282      0.1470      0.2008
   100.000  $    21,594      0.2765      0.2722      0.2112      0.2418

Best Ridge alpha: 0.001
Learned coefficients:
  XGB:      0.3431
  LGBM:     0.3673
  ET:       0.1198
  CatBoost: 0.1725
  Intercept: -0.0349

OOF meta-train RMSE (optimistic): $21,552
v15 OOF RMSE for comparison:        $21,593


---
## 12. Generate Submission v9

In [ ]:
final_log  = meta_model.predict(meta_X_test)
final_pred = np.expm1(final_log)

sample_sub = pd.read_csv('../../outputs/submissions/sample_sub_reg.csv')
sub_v16 = pd.DataFrame({'Id': test['id'], 'Predicted': final_pred})
sub_v16 = sub_v16.set_index('Id').reindex(sample_sub['Id']).reset_index()

out = '../../outputs/submissions/sub_v16_catboost.csv'
sub_v16.to_csv(out, index=False)
print(f'Saved: {out}')
print(f'Shape: {sub_v16.shape}')
print(f'Price range: ${sub_v16.Predicted.min():,.0f} – ${sub_v16.Predicted.max():,.0f}')
print(f'Price mean:  ${sub_v16.Predicted.mean():,.0f}')
print(f'\nBaselines — v15 OOF: $21,593 | v12 OOF (CatBoost on v9 features): $21,578')

---
## 13. 100% Data Retrain

Retrain all 4 base models on the **full 150K training set**. Ridge meta-learner is unchanged.

- OOF RMSE stays $21,552 (Ridge trained on OOF predictions as before)
- Kaggle score improves because test predictions come from models trained on 100% of data
- Full-data group maps (Section 3) applied to X_full — consistent with fold-level encoding strategy

In [ ]:
# Build X_full: apply full-data group maps (not fold-level maps)
FULL_MAPS = {
    'town':           full_town_map,
    'postal_sector':  full_sector_map,
    'flat_model':     full_model_map,
    'town_flat_type': full_tf_map,
    'Tranc_Year':     full_year_map,
}
X_full = X.copy()
for group_col, price_col, _ in ENCODE_PAIRS:
    X_full[price_col] = train_fe[group_col].map(FULL_MAPS[group_col]).fillna(global_price_med).values

# Fit preprocessor on full training data
pre_full = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), cat_cols)
])
pre_full.fit(X_full)
Xf_arr = pre_full.transform(X_full)
Xt_arr = pre_full.transform(X_test)

# Retrain all 4 models on 100% of training data
print('Retraining on 100% of data (150K rows)...')
xgb_f  = XGBRegressor(**XGB_PARAMS);         xgb_f.fit(Xf_arr,  y_log); print('  XGB done')
lgbm_f = LGBMRegressor(**LGBM_PARAMS);       lgbm_f.fit(Xf_arr, y_log); print('  LGBM done')
et_f   = ExtraTreesRegressor(**ET_PARAMS);   et_f.fit(Xf_arr,   y_log); print('  ET done')
cat_f  = CatBoostRegressor(**CAT_PARAMS);    cat_f.fit(Xf_arr,  y_log); print('  CAT done')

# Pass 100% retrain predictions through existing Ridge meta-learner
meta_test_full = np.column_stack([
    xgb_f.predict(Xt_arr),
    lgbm_f.predict(Xt_arr),
    et_f.predict(Xt_arr),
    cat_f.predict(Xt_arr)
])
final_pred_full = np.expm1(meta_model.predict(meta_test_full))

# Save submission
sample_sub_fd = pd.read_csv('../../outputs/submissions/sample_sub_reg.csv')
sub_full = pd.DataFrame({'Id': test['id'], 'Predicted': final_pred_full})
sub_full = sub_full.set_index('Id').reindex(sample_sub_fd['Id']).reset_index()
out_full = '../../outputs/submissions/sub_v16_fulldata.csv'
sub_full.to_csv(out_full, index=False)
print(f'\nSaved: {out_full}')
print(f'Price range: ${sub_full.Predicted.min():,.0f} – ${sub_full.Predicted.max():,.0f}')
print(f'Price mean:  ${sub_full.Predicted.mean():,.0f}')
print(f'\nExpected Kaggle improvement over OOF submission: $50–100')

---
## 13. Summary

### Changes vs v8
| Change | Why | Expected Gain |
|---|---|---|
| `postal_sector_median_price` OOF | Replaced 9,125-level ordinal noise with a real geographic price signal | ~200–400 |
| `flat_model_median_price` OOF | Type S2 vs Improved price spread is ~$550K — direct signal | ~50–150 |
| `block_num` numeric | Replaced 2,514-level ordinal noise | ~50–100 |
| RandomizedSearchCV wider grid (40 trials) for XGB+LGBM | More combinations searched than v8's 30-trial grid | ~50–150 |

### Score tracker
| Version | Model | OOF RMSE | Kaggle |
|---|---|---|---|
| v6 | Log blend + OOF encoding | $21,818 | 22,124 |
| v7 | Stack XGB+LGBM | $21,841 | 21,906 |
| v8 | Stack XGB+LGBM+ET | _(see above)_ | 21,804.67 |
| **v9** | **Wider grid + richer encoding** | **_(run above)_** | **_(submit)_** |

### Key questions after running
- Did `postal_sector_median_price` reduce the OOF RMSE by >100? → confirms geographic signal value
- Are all Ridge coefficients positive and >0.05? → confirms 3-model diversity preserved
- Did the wider search grid find materially different params vs v8?

### Next steps if v9 improves
- [ ] Add CatBoost as 4th base model (native categorical handling, different architecture)
- [ ] Try `planning_area` OOF encoding (finer than `town`, coarser than `postal_sector`)
- [ ] Explore Optuna (Bayesian HPO) for even smarter hyperparameter search